# 01 · Baseline — Retrieval sem Fine-Tuning

Pipeline:
1. Carrega modelo pré-treinado `msmarco-distilbert-base-v4`
2. Gera embeddings de todos os documentos
3. Indexa com **FAISS** (busca por similaridade de cosseno)
4. Para cada query, recupera os top-K documentos
5. Salva `results/run_baseline.txt` no formato TREC

In [2]:
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
print(f'CUDA version    : {torch.version.cuda}')
print(f'GPU count       : {torch.cuda.device_count()}')

PyTorch version : 2.6.0+cu124
CUDA available  : True
CUDA version    : 12.4
GPU count       : 1


## 1 · Instalação

In [3]:
!py -m pip install sentence-transformers faiss-cpu tqdm

## 2 · Imports e configuração

In [4]:
import json, pickle
from pathlib import Path
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

MODEL_NAME  = 'sentence-transformers/all-MiniLM-L6-v2'
TOP_K       = 1000
BATCH_SIZE  = 512   # GPU aguenta lotes maiores; ajuste se der OOM

DATA_DIR    = Path('data')
RESULTS_DIR = Path('results')
MODELS_DIR  = Path('models/baseline')
RESULTS_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

CORPUS_PATH  = DATA_DIR / 'corpus.jsonl'
QUERIES_PATH = DATA_DIR / 'queries.jsonl'
RUN_PATH     = RESULTS_DIR / 'run_baseline.txt'
INDEX_PATH   = MODELS_DIR / 'faiss.index'
DOCIDS_PATH  = MODELS_DIR / 'doc_ids.pkl'

Dispositivo: cuda
GPU: NVIDIA GeForce RTX 3080


## 3 · Funções auxiliares

In [5]:
import time

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f]

def batch_encode(model, texts, batch_size=256, desc='Encoding'):
    t0 = time.time()
    print(f'{desc}: {len(texts):,} textos | batch_size={batch_size}')
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,   # barra por batch
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype('float32')
    elapsed = time.time() - t0
    print(f'{desc} concluído em {elapsed/60:.1f} min  |  shape={embeddings.shape}')
    return embeddings

## 4 · Carregamento dos dados

In [6]:
corpus  = load_jsonl(CORPUS_PATH)
queries = load_jsonl(QUERIES_PATH)
print(f'Corpus : {len(corpus):,} documentos')
print(f'Queries: {len(queries):,} consultas')

Corpus : 369,721 documentos
Queries: 1,444 consultas


## 5 · Carregamento do modelo

In [7]:
print(f'Carregando modelo: {MODEL_NAME}')
model = SentenceTransformer(MODEL_NAME, device=device)
print(f'Modelo carregado! (device={device})')

Carregando modelo: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Modelo carregado! (device=cuda)


## 6 · Geração de embeddings e indexação FAISS

In [8]:
if INDEX_PATH.exists() and DOCIDS_PATH.exists():
    print('Cache encontrado. Carregando índice FAISS...')
    index   = faiss.read_index(str(INDEX_PATH))
    doc_ids = pickle.load(open(DOCIDS_PATH, 'rb'))
    print(f'Índice carregado: {index.ntotal:,} vetores')
else:
    texts   = [d['text']   for d in corpus]
    doc_ids = [d['doc_id'] for d in corpus]

    embeddings = batch_encode(model, texts, batch_size=BATCH_SIZE, desc='Documentos')

    dim   = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)

    print('Indexando no FAISS...')
    CHUNK = 50_000
    for start in tqdm(range(0, len(embeddings), CHUNK), desc='FAISS add'):
        index.add(embeddings[start:start + CHUNK])

    faiss.write_index(index, str(INDEX_PATH))
    pickle.dump(doc_ids, open(DOCIDS_PATH, 'wb'))
    print(f'Índice salvo → {INDEX_PATH}  ({index.ntotal:,} vetores, dim={dim})')

Documentos: 369,721 textos | batch_size=512


Batches:   0%|          | 0/723 [00:00<?, ?it/s]

Documentos concluído em 5.3 min  |  shape=(369721, 384)
Indexando no FAISS...


FAISS add: 100%|██████████| 8/8 [00:00<00:00, 17.19it/s]


Índice salvo → models\baseline\faiss.index  (369,721 vetores, dim=384)


## 7 · Retrieval — top-K por query

In [9]:
print(f'Gerando embeddings das queries...')
q_texts = [q['text'] for q in queries]
q_embs  = batch_encode(model, q_texts, batch_size=BATCH_SIZE, desc='Queries')

print(f'Buscando top-{TOP_K} documentos por query...')
scores, indices = index.search(q_embs, TOP_K)
print('Busca concluída!')

Gerando embeddings das queries...
Queries: 1,444 textos | batch_size=512


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Queries concluído em 0.0 min  |  shape=(1444, 384)
Buscando top-1000 documentos por query...
Busca concluída!


## 8 · Salvar run no formato TREC

In [10]:
with open(RUN_PATH, 'w', encoding='utf-8') as f:
    for i, query in enumerate(queries):
        qid = query['query_id']
        for rank, (idx, score) in enumerate(zip(indices[i], scores[i]), start=1):
            did = doc_ids[idx]
            f.write(f'{qid}\tQ0\t{did}\t{rank}\t{score:.6f}\tbaseline\n')

print(f'Run salva → {RUN_PATH}')
print(f'Total de linhas: {sum(1 for _ in open(RUN_PATH)):,}')

Run salva → results\run_baseline.txt
Total de linhas: 1,444,000


## 9 · Preview da run

In [11]:
with open(RUN_PATH) as f:
    lines = [f.readline() for _ in range(5)]

print('Primeiras linhas do run_baseline.txt:')
print('qid  Q0  doc_id  rank  score  run_name')
print('-' * 60)
for l in lines:
    print(l.strip())

Primeiras linhas do run_baseline.txt:
qid  Q0  doc_id  rank  score  run_name
------------------------------------------------------------
123839	Q0	806300	1	0.573176	baseline
123839	Q0	123839	2	0.499044	baseline
123839	Q0	1668368	3	0.481551	baseline
123839	Q0	2320659	4	0.457058	baseline
123839	Q0	434612	5	0.442722	baseline
